# Closed-loop adaptive optics in modal space

This tutorial demonstrates a simple closed-loop AO simulation using the new `wavefront_control` module of HCIPy. We will build a controller and an estimator, combine them, and show how they can be used to drive a deformable mirror in a feedback loop.


In [ ]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt


## An integral controller

The `IntegralController` is the workhorse of AO: it accumulates measurements over time and produces a control signal proportional to their sum. A multiplicative leak prevents windup. Let us create one and look at its step response.


In [ ]:
controller = IntegralController(gain_matrix=0.3, leak=1.0)

control_signals = []
for _ in range(30):
    u = controller.compute(np.array([1.0]), dt=1)
    control_signals.append(u)

plt.plot(control_signals)
plt.xlabel("Iteration")
plt.ylabel("Control signal")
plt.title("IntegralController: step response")
plt.show()


If we add a leak, the integrator approaches a steady state instead of growing without bound.


In [ ]:
controller_leaky = IntegralController(gain_matrix=0.3, leak=0.9)

signals_leaky = []
for _ in range(30):
    u = controller_leaky.compute(np.array([1.0]), dt=1)
    signals_leaky.append(u)

plt.plot(signals_leaky, label="leak=0.9")
plt.plot(control_signals, label="leak=1.0")
plt.xlabel("Iteration")
plt.ylabel("Control signal")
plt.legend()
plt.title("IntegralController: effect of leak")
plt.show()


## Setting up the AO system

We simulate an 8-meter telescope with a deformable mirror controlled in 10 Zernike modes. The DM applies a phase correction, and our synthetic wavefront sensor measures the residual with noise. We use a `FraunhoferPropagator` to evaluate the PSF.


In [ ]:
wavelength = 1e-6
telescope_diameter = 8.0

pupil_grid = make_pupil_grid(128, telescope_diameter)
focal_grid = make_focal_grid(4, 16)
prop = FraunhoferPropagator(pupil_grid, focal_grid)

aperture = make_circular_aperture(telescope_diameter)(pupil_grid)
modes = make_zernike_basis(11, telescope_diameter, pupil_grid, starting_mode=2)
dm = DeformableMirror(modes)

wf_ref = Wavefront(aperture, wavelength)
psf_perfect = prop(wf_ref).power
print(f"Number of modes: {dm.num_actuators}")


## Closing the loop

We start with a fixed aberrated DM shape. The WFS measures the current DM state, and the `IntegralController` with negative gain integrates the residual to drive the DM toward zero. The Strehl ratio improves as the controller converges.


In [ ]:
gain = 0.3
wfs_noise = 0.02 * wavelength
controller = IntegralController(gain_matrix=-gain, leak=1.0)

dm.actuators = np.ones(dm.num_actuators) * 0.3 * wavelength
psf_before = prop(dm.forward(wf_ref)).power

strehl_history = []
for iteration in range(30):
    measurement = dm.actuators + np.random.normal(0, wfs_noise, size=dm.num_actuators)
    dm.actuators = controller.compute(measurement, dt=1)
    psf = prop(dm.forward(wf_ref)).power
    strehl_history.append(psf.max() / psf_perfect.max())

plt.plot(strehl_history)
plt.xlabel("Iteration")
plt.ylabel("Strehl ratio")
plt.title("Closed-loop AO with IntegralController")
plt.grid(True)
plt.show()


In [ ]:
psf_after = prop(dm.forward(wf_ref)).power

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
imshow_psf(psf_before)
plt.title("Before correction")
plt.subplot(1, 3, 2)
imshow_psf(psf_after)
plt.title("After correction")
plt.subplot(1, 3, 3)
imshow_psf(psf_perfect)
plt.title("Perfect (diffraction-limited)")
plt.show()


## Adding a Kalman filter

The `KalmanFilter` can be used to estimate the state of a dynamical system from noisy measurements. Here we model the evolution of the Zernike mode coefficients as a random walk and use the Kalman filter to track them.


In [ ]:
n_modes = dm.num_actuators
A_kf = np.eye(n_modes)
B_kf = None
C_kf = np.eye(n_modes)
Q_kf = np.eye(n_modes) * (0.01 * wavelength) ** 2
R_kf = np.eye(n_modes) * (wfs_noise) ** 2

kf = KalmanFilter(A_kf, B_kf, C_kf, Q_kf, R_kf)
print(f"Initial P diagonal: {np.diag(kf.P)[:3]}")


## LQR control

The `LQRController` implements state feedback with a pre-computed gain matrix $K$. Here we solve the discrete-time algebraic Riccati equation for a double-integrator model and show how the LQR controller produces a control signal from a state estimate.


In [ ]:
from scipy.linalg import solve_discrete_are

dt = 0.1
A_d = np.array([[1, dt], [0, 1]])
B_d = np.array([[0], [dt]])
Q_lqr = np.diag([1.0, 0.1])
R_lqr = np.array([[0.01]])

P = solve_discrete_are(A_d, B_d, Q_lqr, R_lqr)
K = np.linalg.inv(B_d.T @ P @ B_d + R_lqr) @ (B_d.T @ P @ A_d)

lqr = LQRController(K)
x = np.array([1.0, 0.0])
print(f"State: {x}")
print(f"LQR control: {lqr.compute(x)}")


## Summary

We have used the `wavefront_control` module to build a closed-loop AO system. The `IntegralController` provides the core feedback, the `KalmanFilter` can be used for state estimation, and the `LQRController` implements state-feedback control. Each class is pluggable and can be combined to suit the needs of the application.
